# Create Virtual Icechunk Store from ERA5 Single-Levels Monthly-Mean NetCDF files

ERA5 single-levels data from [CDS](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries) is not hosted as
gridded NetCDF on a public anonymous S3, so it cannot be virtualized in-place the way Copernicus Marine files can.
Instead this notebook:

1. **Downloads** monthly-mean ERA5 single-level NetCDF files from CDS via `cdsapi` (one file per month).
2. **Stages** the files on protocoast S3 (`s3://protocoast-data/era5-sl-netcdf/`).
3. **Virtualizes** each file with VirtualiZarr and concatenates along `time`.
4. **Writes** an Icechunk repository whose virtual chunks point back at those S3 files.
5. **Verifies** the store by opening it with xarray and plotting a 2-m temperature field.

**Variables** (configurable): `2m_temperature`, `mean_sea_level_pressure`, `10m_u_component_of_wind`, `10m_v_component_of_wind`  
**Default time range**: 2020-01 — 2020-12 (12 files, ~40 MB total)  
**Source**: CDS `reanalysis-era5-single-levels-monthly-means` (monthly-averaged reanalysis on a 0.25° global grid)

> **CDS account required** — register at https://cds.climate.copernicus.eu and follow the steps in the _Credentials_ section below.

In [ ]:
# Install cdsapi if not already present
import importlib
if importlib.util.find_spec('cdsapi') is None:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'cdsapi'])

In [ ]:
import warnings
import os
import pathlib
import itertools

import cdsapi
import fsspec
import s3fs
import pandas as pd
import xarray as xr
from dotenv import load_dotenv

import icechunk
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import HDFParser
from obspec_utils.registry import ObjectStoreRegistry
from obstore.store import S3Store

warnings.filterwarnings('ignore', category=UserWarning)

In [ ]:
# ── Variables to download ──────────────────────────────────────────────────────
VARIABLES = [
    '2m_temperature',
    'mean_sea_level_pressure',
    '10m_u_component_of_wind',
    '10m_v_component_of_wind',
]

# ── Time range (inclusive) ─────────────────────────────────────────────────────
YEAR_START, YEAR_END   = 2020, 2020   # increase for a longer record
MONTH_START, MONTH_END = 1,    12

# Build list of (year, month) tuples
ALL_MONTHS = [
    (y, m)
    for y in range(YEAR_START, YEAR_END + 1)
    for m in range(1, 13)
    if (y, m) >= (YEAR_START, MONTH_START) and (y, m) <= (YEAR_END, MONTH_END)
]
print(f'Time steps to download: {len(ALL_MONTHS)}')

# ── Local staging directory ────────────────────────────────────────────────────
LOCAL_DIR = pathlib.Path('era5_sl_downloads')
LOCAL_DIR.mkdir(exist_ok=True)

# ── Protocoast S3 ──────────────────────────────────────────────────────────────
_ = load_dotenv(f'{os.environ["HOME"]}/dotenv/protocoast.env', override=True)
storage_endpoint = os.environ['ENDPOINT_URL']
storage_bucket   = 'protocoast-data'
netcdf_prefix    = 'era5-sl-netcdf'             # where staged NetCDF files live
icechunk_name    = 'era5-sl-icechunk-v1'         # Icechunk repo name

netcdf_s3_prefix = f's3://{storage_bucket}/{netcdf_prefix}/'

## Step 0 — CDS credentials

1. Log in at https://cds.climate.copernicus.eu
2. Go to your profile → _API key_ and copy your key (UUID format).
3. Create `~/.cdsapirc` with:

```
url: https://cds.climate.copernicus.eu/api
key: <your-api-key>
```

Or set the environment variable `CDSAPI_KEY=<your-api-key>` before running this notebook.
The cell below will raise an error with instructions if the credentials are missing.

In [ ]:
cds = cdsapi.Client(quiet=True)
print('CDS client ready.')

## Step 1 — Download monthly ERA5 NetCDF files from CDS

One file per month; each file contains all `VARIABLES` at that month's global 0.25° grid.
Files already present on disk are skipped so the cell is safe to re-run.

In [ ]:
local_files = []

for year, month in ALL_MONTHS:
    fname = LOCAL_DIR / f'era5_sl_{year}_{month:02d}.nc'
    local_files.append(fname)

    if fname.exists():
        print(f'  skip  {fname.name}  (already on disk)')
        continue

    print(f'  fetch {fname.name} ...', end=' ', flush=True)
    cds.retrieve(
        'reanalysis-era5-single-levels-monthly-means',
        {
            'product_type': 'monthly_averaged_reanalysis',
            'variable'    : VARIABLES,
            'year'        : str(year),
            'month'       : f'{month:02d}',
            'time'        : '00:00',
            'data_format' : 'netcdf',
        },
        fname,
    )
    print(f'{fname.stat().st_size / 1e6:.1f} MB')

print(f'\nAll {len(local_files)} files ready in {LOCAL_DIR}/')

## Step 2 — Inspect one file

Confirm the variable names, dimensions, and coordinates before virtualizing.

In [ ]:
ds_sample = xr.open_dataset(local_files[0])
print(ds_sample)
ds_sample

## Step 3 — Upload files to protocoast S3

Files are uploaded to `s3://protocoast-data/era5-sl-netcdf/` so the Icechunk virtual
chunks can reference a stable, cloud-accessible URL.  Already-uploaded files are skipped.

In [ ]:
fs_proto = s3fs.S3FileSystem(endpoint_url=storage_endpoint)

s3_files = []
for local in local_files:
    s3_path = f'{storage_bucket}/{netcdf_prefix}/{local.name}'
    s3_url  = f's3://{s3_path}'
    s3_files.append(s3_url)

    if fs_proto.exists(s3_path):
        print(f'  skip  {s3_url}  (already on S3)')
    else:
        print(f'  upload {local.name} → {s3_url} ...', end=' ', flush=True)
        fs_proto.put(str(local), s3_path)
        print('done')

print(f'\n{len(s3_files)} files staged at {netcdf_s3_prefix}')

## Step 4 — Configure Icechunk repository

Both the Icechunk store **and** the source NetCDF files live on protocoast S3, so the
virtual chunk container uses `None` for credentials (inherit from environment).

In [ ]:
# Icechunk store on protocoast
storage = icechunk.s3_storage(
    bucket=storage_bucket,
    prefix=f'icechunk/{icechunk_name}',
    from_env=True,
    endpoint_url=storage_endpoint,
    region='not-used',
    force_path_style=True,
)

config = icechunk.RepositoryConfig.default()

# Virtual chunk container — NetCDF chunks live at the same protocoast S3 endpoint
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=netcdf_s3_prefix,
        store=icechunk.s3_store(
            region='not-used',
            s3_compatible=True,
            force_path_style=True,
            endpoint_url=storage_endpoint,
            from_env=True,
        ),
    )
)

# Authorize virtual chunk access using env credentials (None = inherit from env)
credentials = icechunk.containers_credentials(
    {netcdf_s3_prefix: None}
)

# VirtualiZarr object-store registry (same store, keyed by s3:// prefix)
proto_store = S3Store(
    bucket=storage_bucket,
    endpoint_url=storage_endpoint,
    region='not-used',
    skip_signature=False,
    aws_access_key_id=os.environ['AWS_ACCESS_KEY_ID'],
    aws_secret_access_key=os.environ['AWS_SECRET_ACCESS_KEY'],
)
registry = ObjectStoreRegistry({netcdf_s3_prefix: proto_store})
parser   = HDFParser()

print('Icechunk config ready.')
print(f'  Store   : s3://{storage_bucket}/icechunk/{icechunk_name}')
print(f'  NetCDF  : {netcdf_s3_prefix}')

## Step 5 — Virtualize files and write to Icechunk

Each monthly file becomes a virtual dataset; they are concatenated along `time`
before being written in a single Icechunk commit.  The time coordinate is loaded
eagerly so xarray can concatenate correctly.

In [ ]:
print('Virtualizing files...')
vds_list = [
    open_virtual_dataset(url, parser=parser, registry=registry, loadable_variables=['time'])
    for url in s3_files
]

ds_virtual = xr.concat(
    vds_list, dim='time', coords='minimal', compat='override', combine_attrs='override'
)
print(ds_virtual)

In [ ]:
# Delete any pre-existing repo at this path, then create fresh
icechunk_prefix = f'icechunk/{icechunk_name}'
if fs_proto.exists(f'{storage_bucket}/{icechunk_prefix}'):
    fs_proto.rm(f'{storage_bucket}/{icechunk_prefix}', recursive=True)
    print(f'Deleted existing repo at s3://{storage_bucket}/{icechunk_prefix}')

repo    = icechunk.Repository.create(storage, config, authorize_virtual_chunk_access=credentials)
session = repo.writable_session('main')

t0 = pd.Timestamp(ds_virtual.time.values[0]).date()
t1 = pd.Timestamp(ds_virtual.time.values[-1]).date()

print(f'Writing {len(ds_virtual.time)} time steps to Icechunk...')
ds_virtual.virtualize.to_icechunk(session.store)

msg = f'Initialized from {len(s3_files)} ERA5 monthly-mean files ({t0} to {t1})'
snapshot_id = session.commit(msg)
print(f'Committed: "{msg}"  (snapshot {snapshot_id})')

## Step 6 — Verify — open from Icechunk and read actual data

In [ ]:
history = repo.ancestry(branch='main')
latest  = next(history)
print(f'Latest commit [{latest.written_at}]: {latest.message}')

session_ro = repo.readonly_session('main')
ds_check   = xr.open_zarr(session_ro.store, consolidated=False, chunks={})
print(ds_check)
ds_check

In [ ]:
import hvplot.xarray

# Load one 2-m temperature field to confirm virtual chunk access works
t2m = ds_check['t2m'].isel(time=0).load() - 273.15   # K → °C
t2m.attrs['units'] = '°C'

t2m.hvplot(
    x='longitude', y='latitude',
    rasterize=True,
    geo=True,
    tiles='OSM',
    cmap='RdBu_r',
    clabel='2-m Temperature (°C)',
    frame_width=800,
    frame_height=450,
    title=f'ERA5 2-m Temperature — {pd.Timestamp(ds_check.time.values[0]).strftime("%B %Y")}',
)